# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *56* |
| **Integrantes** | *Paula Andrea Celis Cano* |
| **Caso de estudio** | *(Wanderbricks)* |
| **Fecha de entrega** | domingo 20 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*Qué necesidad de infraestructura plantea el caso y qué debe soportar el entorno.*

---
## 2. Descripción de los datos

*Qué va a vivir en esta infraestructura: volumen esperado, frecuencia de actualización
y quién la va a consumir.*

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura

*Fuentes → ingesta → almacenamiento → procesamiento → consumo.
Señalar explícitamente qué capa administra el proveedor y cuál el equipo.
Insertar la imagen o usar un diagrama en texto.*

```
[ fuentes ] → [ ingesta ] → [ almacenamiento ] → [ procesamiento ] → [ consumo ]
```

### 3.2 Matriz de roles

*Qué puede hacer cada rol sobre cada capa en un entorno real.*

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | | | |
| Ingeniero de datos | | | |
| Administrador | | | |

### 3.3 Especificación del equivalente IaaS

*Se diseña, no se implementa. Máquinas y dimensionamiento, sistema operativo,
software a instalar, red, almacenamiento, y estimación del esfuerzo de puesta
en marcha y de operación.*

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS | SaaS |
|---|---|---|---|
| Control | | | |
| Tiempo hasta el primer resultado | | | |
| Esfuerzo operativo | | | |
| Costo | | | |
| Escalabilidad | | | |
| Gobierno | | | |

**Conclusión:** *cuál conviene a este caso y por qué.*

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# ==========================================
# EA2 - Organización del entorno Wanderbricks
# ==========================================

CATALOGO = "bigdata_grupo35"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.bronce")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.plata")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.oro")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronce.datos_crudos")

print("✅ Organización de Wanderbricks creada")

In [0]:
# Mostrar los esquemas creados
display(
    spark.sql(f"SHOW SCHEMAS IN {CATALOGO}")
)

In [0]:
from pyspark.sql import functions as F

bookings = spark.table("samples.wanderbricks.bookings")

bookings_bronze = (
    bookings
    .withColumn("_ingesta_ts", F.current_timestamp())
    .withColumn("_origen", F.lit("samples.wanderbricks.bookings"))
)

bookings_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bigdata_grupo35.bronce.bookings")

print("✅ Tabla Bronze bookings creada")

In [0]:
display(
    spark.table(f"{CATALOGO}.bronce.bookings")
    .limit(10)
)

In [0]:
cantidad = spark.table(
    f"{CATALOGO}.bronce.bookings"
).count()

print(f"Bronze bookings: {cantidad:,} filas")

In [0]:
from pyspark.sql import functions as F

bookings_bronze = spark.table(
    "bigdata_grupo35.bronce.bookings"
)

bookings_plata = (
    bookings_bronze
    .filter(F.col("booking_id").isNotNull())
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("property_id").isNotNull())
    .withColumn(
        "dias_estadia",
        F.datediff(
            F.col("check_out"),
            F.col("check_in")
        )
    )
)

bookings_plata.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo35.plata.bookings"
    )

print("✅ Tabla Plata bookings creada")

In [0]:
bookings_plata = spark.table(
    "bigdata_grupo35.plata.bookings"
)

resumen_oro = (
    bookings_plata
    .groupBy("status")
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(
            F.sum("total_amount"), 2
        ).alias("ingresos_totales")
    )
)

resumen_oro.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo35.oro.resumen_reservas"
    )

print("✅ Tabla Oro creada")

In [0]:
bookings_plata = spark.table(
    "bigdata_grupo35.plata.bookings"
)

resumen_oro = (
    bookings_plata
    .groupBy("status")
    .agg(
        F.count("*").alias("cantidad_reservas"),
        F.round(
            F.sum("total_amount"), 2
        ).alias("ingresos_totales")
    )
)

resumen_oro.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "bigdata_grupo35.oro.resumen_reservas"
    )

print("✅ Tabla Oro creada")

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
CATALOGO = "bigdata_grupo35"

# Permiso para usar el catálogo
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `account users`")

# Permisos sobre ORO
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.oro TO `account users`")
spark.sql(f"GRANT SELECT ON SCHEMA {CATALOGO}.oro TO `account users`")

# Permisos sobre PLATA
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.plata TO `account users`")
spark.sql(f"GRANT MODIFY ON SCHEMA {CATALOGO}.plata TO `account users`")

print("✅ Permisos otorgados correctamente")

In [0]:
print("Permisos sobre ORO:")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.oro"))

print("Permisos sobre PLATA:")
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.plata"))

### 4.3 Linaje

Evidencia del linaje: La tabla de Oro se genera a partir de la tabla de Plata, que a su vez proviene de los datos almacenados en Bronze. El linaje permite visualizar el recorrido de los datos dentro de la arquitectura del Lakehouse.

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

In [0]:
# TODO: código de las tareas que ejecuta el Job

---
## 5. Resultados

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas